# Cross-Dataset Phishing Website Detection
## Notebook 1: Data Loading, EDA and Preprocessing

**What this notebook does**
1. Loads PhiUSIIL (primary) and UCI Phishing Websites (secondary) datasets
2. Harmonizes labels so both use `is_phishing` (1 = phishing, 0 = legitimate)
3. EDA: class balance, distributions, correlation heatmaps
4. Cleans data: drops text columns, handles duplicates, identifies skewed columns
5. Saves clean files to `data/processed/` for subsequent notebooks

**Run order:** Top to bottom. If `ucimlrepo` fails, loaders fall back to local files in `data/raw/`.

## 0. Setup

In [ ]:
!pip install ucimlrepo xgboost shap -q

In [ ]:
import json, math, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

SEED = 42
np.random.seed(SEED)

# ── Detect environment: Colab vs local ──
# On Colab, set USE_DRIVE = True after uploading the project to Google Drive
USE_DRIVE = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/phishing-project')
elif IN_COLAB:
    BASE = Path('/content/phishing-project')
else:
    # Local: notebook lives in notebooks/, project root is one level up
    BASE = Path('..').resolve()

# Add project root to Python path so we can import src/ modules
sys.path.insert(0, str(BASE))

from src.data_utils import (
    project_paths, load_phiusiil_raw, load_uci_raw,
    harmonize_labels, drop_text_columns, dedupe_report,
    identify_skewed_columns, CLASS_NAMES,
)

dirs = project_paths(BASE)
RAW_DIR  = dirs['raw']
PROC_DIR = dirs['processed']
FIG_DIR  = dirs['figures']
TAB_DIR  = dirs['tables']

def savefig(name):
    """Save current figure to results/figures/ at 150 dpi."""
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

print("Folders ready under:", BASE.resolve())

## Phase 1: Data Loading and Understanding

Both loaders try the `ucimlrepo` package first. If that fails, upload the files to `data/raw/`:
- PhiUSIIL: `PhiUSIIL_Phishing_URL_Dataset.csv`
- UCI Phishing Websites: `Training Dataset.arff`

In [ ]:
# Load both datasets using src/data_utils.py helper functions
# WHY we use helper functions: if the UCI API is down, they automatically
# fall back to local CSV/ARFF files, making the notebook reproducible.
phi_raw = load_phiusiil_raw(RAW_DIR)
uci_raw = load_uci_raw(RAW_DIR)

print("\nPhiUSIIL shape:", phi_raw.shape)
print("UCI Phishing shape:", uci_raw.shape)

In [ ]:
# First look: PhiUSIIL
# WHY: verify columns, dtypes, and check for missing values before any processing
display(phi_raw.head(3))
print("\nData types:")
print(phi_raw.dtypes.value_counts())
print("\nMissing values (PhiUSIIL):", int(phi_raw.isnull().sum().sum()))
print("\nAll columns:", phi_raw.columns.tolist())

In [ ]:
# First look: UCI Phishing Websites
display(uci_raw.head(3))
print("\nData types:")
print(uci_raw.dtypes.value_counts())
print("\nMissing values (UCI):", int(uci_raw.isnull().sum().sum()))
print("\nAll columns:", uci_raw.columns.tolist())

### 1.1 Harmonize the labels

The two datasets use **different label conventions**:

| Dataset | Original coding |
|---|---|
| PhiUSIIL | `label`: 1 = legitimate, 0 = phishing |
| UCI Phishing Websites | `Result`: 1 = legitimate, −1 = phishing |

We create one shared column, `is_phishing` (**1 = phishing, 0 = legitimate**), in both.

In [ ]:
# harmonize_labels() creates 'is_phishing' and renames the original target
# to 'label_raw' so we can always double-check the mapping.
phi, uci = harmonize_labels(phi_raw, uci_raw)

# Sanity check against published class counts
print("PhiUSIIL phishing count:", int(phi["is_phishing"].sum()), "(expected ~100,945)")
print("UCI phishing count     :", int(uci["is_phishing"].sum()), "(expected ~4,898)")

## Phase 2: Exploratory Data Analysis (EDA)

### 2.1 Class distribution
WHY: check if the data is imbalanced, which would require stratified splits and possibly class weights.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, df) in zip(axes, [("PhiUSIIL", phi), ("UCI Phishing", uci)]):
    counts = df["is_phishing"].map(CLASS_NAMES).value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=ax)
    ax.set_title(f"{name} class distribution")
    ax.set_ylabel("Records")
    for i, v in enumerate(counts.values):
        ax.text(i, v, f"{v:,}\n({v/len(df):.1%})", ha="center", va="bottom")
    ax.set_ylim(0, counts.max() * 1.18)
plt.tight_layout()
savefig("01_class_distribution")
plt.show()

print("Both datasets are roughly balanced (40-57% per class) — no SMOTE needed.")

### 2.2 PhiUSIIL: feature distributions by class
Values above the 99th percentile are clipped so extreme outliers don't flatten the histogram.

In [ ]:
PHI_PLOT_FEATS = ["URLLength", "DomainLength", "NoOfSubDomain",
                  "LetterRatioInURL", "URLSimilarityIndex", "LineOfCode"]
feats = [c for c in PHI_PLOT_FEATS if c in phi.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), feats):
    hi = phi[col].quantile(0.99)
    keep = phi[col] <= hi
    tmp = pd.DataFrame({col: phi.loc[keep, col],
                        "Class": phi.loc[keep, "is_phishing"].map(CLASS_NAMES)})
    sns.histplot(data=tmp, x=col, hue="Class", bins=40, stat="density",
                 common_norm=False, element="step", ax=ax)
    ax.set_title(col)
for ax in axes.ravel()[len(feats):]:
    ax.axis("off")
plt.tight_layout()
savefig("02_phiusiil_feature_distributions")
plt.show()

In [ ]:
# Boxplots for the same features — spot outliers and class separation
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), feats):
    tmp = pd.DataFrame({col: phi[col], "Class": phi["is_phishing"].map(CLASS_NAMES)})
    sns.boxplot(data=tmp, x="Class", y=col, ax=ax, showfliers=False)
    ax.set_title(col)
for ax in axes.ravel()[len(feats):]:
    ax.axis("off")
plt.tight_layout()
savefig("03_phiusiil_boxplots")
plt.show()

### 2.3 UCI Phishing: every feature is −1 / 0 / 1
Count plots show which values are typical of phishing vs legitimate sites.

In [ ]:
uci_feat_cols = [c for c in uci.columns if c not in ("label_raw", "is_phishing")]
ncols = 5
nrows = math.ceil(len(uci_feat_cols) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 3.2 * nrows))
for ax, col in zip(axes.ravel(), uci_feat_cols):
    tmp = pd.DataFrame({col: uci[col], "Class": uci["is_phishing"].map(CLASS_NAMES)})
    sns.countplot(data=tmp, x=col, hue="Class", ax=ax)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("")
    ax.set_ylabel("")
    leg = ax.get_legend()
    if leg:
        leg.remove()
for ax in axes.ravel()[len(uci_feat_cols):]:
    ax.axis("off")
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", fontsize=11)
plt.tight_layout()
savefig("04_uci_countplots")
plt.show()

### 2.4 Correlation heatmaps
Look for: (a) blocks of strongly correlated features (candidates for removal),
(b) features strongly tied to `is_phishing`,
(c) any PhiUSIIL feature with correlation close to 1.0 (possible leakage).

In [ ]:
# PhiUSIIL: numeric columns only (text columns excluded automatically)
phi_num = phi.select_dtypes(include=[np.number]).drop(columns=["label_raw"])
corr_phi = phi_num.corr()

mask = np.triu(np.ones_like(corr_phi, dtype=bool))
plt.figure(figsize=(22, 18))
sns.heatmap(corr_phi, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.2, cbar_kws={"shrink": 0.6})
plt.title("PhiUSIIL correlation heatmap (numeric features)", fontsize=16)
plt.tight_layout()
savefig("05_phiusiil_corr_heatmap")
plt.show()

In [ ]:
# UCI: all features are numeric (-1/0/1), so annotation is readable
uci_num = uci.drop(columns=["label_raw"])
corr_uci = uci_num.corr()

mask = np.triu(np.ones_like(corr_uci, dtype=bool))
plt.figure(figsize=(18, 15))
sns.heatmap(corr_uci, mask=mask, annot=True, fmt=".1f", annot_kws={"size": 6},
            cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.2, cbar_kws={"shrink": 0.6})
plt.title("UCI Phishing Websites correlation heatmap", fontsize=16)
plt.tight_layout()
savefig("06_uci_corr_heatmap")
plt.show()

In [ ]:
# Features most correlated with the target (absolute value)
# WHY: identifies which features are most predictive, and flags any
# with suspiciously high correlation (possible data leakage).
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (name, corr) in zip(axes, [("PhiUSIIL", corr_phi), ("UCI Phishing", corr_uci)]):
    top = (corr["is_phishing"].drop("is_phishing")
           .abs().sort_values(ascending=False).head(15))
    sns.barplot(x=top.values, y=top.index, ax=ax)
    ax.set_title(f"{name}: top 15 |correlation| with is_phishing")
    ax.set_xlabel("|correlation|")
plt.tight_layout()
savefig("07_top_target_correlations")
plt.show()

# Flag features with |corr| > 0.9 as potential leakage
phi_high = corr_phi["is_phishing"].drop("is_phishing").abs()
leaky = phi_high[phi_high > 0.9]
if len(leaky) > 0:
    print("\n⚠ PhiUSIIL features with |corr| > 0.9 with target (ablation candidates):")
    print(leaky.sort_values(ascending=False))
else:
    print("\nNo PhiUSIIL features have |corr| > 0.9 with target.")

## Phase 3: Preprocessing and Cleaning

### 3.1 PhiUSIIL: remove text and identifier columns
Raw strings (`URL`, `Domain`, `Title`, `FILENAME`) identify individual websites —
a model would memorize them instead of learning patterns.
`TLD` as text is dropped; the numeric `TLDLegitimateProb` already captures TLD info.

In [ ]:
# drop_text_columns() removes all non-numeric columns plus known text identifiers
phi_clean, dropped_cols = drop_text_columns(phi)
print("Dropped from PhiUSIIL:", dropped_cols)
print("PhiUSIIL after dropping text:", phi_clean.shape)

### 3.2 UCI: check the value coding

In [ ]:
uci_clean = uci.drop(columns=["label_raw"]).astype(int)

uci_feats = uci_clean.drop(columns=["is_phishing"])
print("All UCI features within {-1, 0, 1}:", bool(uci_feats.isin([-1, 0, 1]).all().all()))

# Features that only use two of the three values
two_valued = uci_feats.nunique()[lambda s: s == 2].index.tolist()
print(f"{len(two_valued)} UCI features are two-valued:", two_valued)

### 3.3 Duplicates
Duplicate rows can leak into both train and test splits, inflating scores.
We report exact duplicates AND rows with identical features but **different labels**
(conflicting vectors), then remove exact duplicates.

In [ ]:
phi_clean, phi_dup_stats = dedupe_report(phi_clean, "PhiUSIIL")
uci_clean, uci_dup_stats = dedupe_report(uci_clean, "UCI Phishing")

print("\nClass balance after dedupe:")
print("PhiUSIIL:", phi_clean["is_phishing"].value_counts(normalize=True).round(3).to_dict())
print("UCI     :", uci_clean["is_phishing"].value_counts(normalize=True).round(3).to_dict())

### 3.4 Skewed features (PhiUSIIL)
Count-type features like `LineOfCode` have long right tails.
Tree models don't care, but **LR and SVM** do. We only *list* the columns here;
the actual `log1p` transform goes inside the model Pipeline in Notebook 03,
so nothing leaks between train and test.

In [ ]:
binary_cols, cont_cols, LOG_COLS = identify_skewed_columns(phi_clean)

print(f"{len(binary_cols)} binary columns, {len(cont_cols)} continuous columns")
print(f"{len(LOG_COLS)} columns flagged for log1p (skew > 3, non-negative):")

phi_X = phi_clean.drop(columns=["is_phishing"])
skew = phi_X[cont_cols].skew().sort_values(ascending=False)
display(skew[LOG_COLS].round(1).to_frame("skewness"))

In [ ]:
# Visual check: before vs after log1p on the 3 most skewed columns
# WHY: confirm that log1p actually helps compress the tails
top3 = LOG_COLS[:3]
if top3:
    fig, axes = plt.subplots(2, len(top3), figsize=(5 * len(top3), 7), squeeze=False)
    for j, col in enumerate(top3):
        sns.histplot(phi_X[col], bins=50, ax=axes[0, j]).set_title(f"{col} (raw)")
        sns.histplot(np.log1p(phi_X[col]), bins=50, ax=axes[1, j]).set_title(f"{col} (log1p)")
    plt.tight_layout()
    savefig("08_log_transform_check")
    plt.show()

### 3.5 Save processed data for subsequent notebooks

In [ ]:
phi_clean.to_csv(PROC_DIR / "phiusiil_clean.csv", index=False)
uci_clean.to_csv(PROC_DIR / "uci_phishing_clean.csv", index=False)

meta = {
    "label_convention": "is_phishing: 1 = phishing, 0 = legitimate",
    "phiusiil": {
        "shape": list(phi_clean.shape),
        "dropped_columns": dropped_cols,
        "binary_cols": binary_cols,
        "continuous_cols": cont_cols,
        "log_cols": LOG_COLS,
        "duplicates": phi_dup_stats,
    },
    "uci": {
        "shape": list(uci_clean.shape),
        "features": [c for c in uci_clean.columns if c != "is_phishing"],
        "duplicates": uci_dup_stats,
    },
    "seed": SEED,
}
with open(PROC_DIR / "preprocessing_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:")
for p in sorted(PROC_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

### 3.6 Summary statistics table
Save a table of summary statistics for the report.

In [ ]:
summary = pd.DataFrame({
    "Dataset": ["PhiUSIIL", "UCI Phishing"],
    "Rows (raw)": [phi_dup_stats["rows_before"], uci_dup_stats["rows_before"]],
    "Rows (after dedupe)": [phi_dup_stats["rows_after"], uci_dup_stats["rows_after"]],
    "Features": [phi_clean.shape[1] - 1, uci_clean.shape[1] - 1],
    "Exact duplicates": [phi_dup_stats["exact_duplicates"], uci_dup_stats["exact_duplicates"]],
    "Conflicting vectors": [phi_dup_stats["conflicting_vectors"], uci_dup_stats["conflicting_vectors"]],
    "Phishing %": [
        f"{phi_clean['is_phishing'].mean():.1%}",
        f"{uci_clean['is_phishing'].mean():.1%}",
    ],
})
display(summary)
summary.to_csv(TAB_DIR / "01_dataset_summary.csv", index=False)
print("\nSaved to", TAB_DIR / "01_dataset_summary.csv")

## What's Next

**Notebook 02: Feature Alignment** — Build the mapping table between PhiUSIIL and UCI features,
recode PhiUSIIL values into UCI's −1/0/1 scheme, and produce two identically-columned datasets
for cross-dataset experiments.

**Key leak-free pattern** (used from Notebook 03 onward):
```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([("scale", StandardScaler()),
                 ("clf", LogisticRegression(max_iter=1000))])
# fit() on training data only — scaler never sees test data
```